In [1]:
import random
import re
import pickle
from pathlib import Path
from collections import defaultdict
from functools import partial
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.signal import find_peaks, peak_widths
from scipy.ndimage import uniform_filter1d

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tqdm import tqdm

from sequana import FastA

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model


# Homopolymere 

In [7]:
#Calcule des metriques
def count_homopolymers(seq, min_length=5):
    pattern = re.compile(rf"(A{{{min_length},}}|T{{{min_length},}}|C{{{min_length},}}|G{{{min_length},}})")
    return len(pattern.findall(seq.upper()))

def process_sequence(maseq, window_size=100):
    seq = maseq.sequence.upper()
    X2, X3 = [], []
    print(maseq.name)
    for i in range(0, len(seq)-2, 1):
        window = seq[max(0, i - window_size//2):min(i+window_size//2, len(seq))]
        X2.append(count_homopolymers(window, min_length=2))
        X3.append(count_homopolymers(window, min_length=3))

   
    if len(X2) > 10000: 
        X2 = X2[5000:-5000]
        X3 = X3[5000:-5000]

    df = pd.DataFrame({'X2': X2, 'X3': X3})
    return df

def load_fasta_parallel(fasta_path, window_size=100, max_workers=None):
    f = FastA(fasta_path)
    sequences = list(f)  # charge toutes les séquences en mémoire

   
    func = partial(process_sequence, window_size=window_size)

    data = []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(func, sequences)

        for df in results:
            data.append(df)

    return data

# Exemple d’utilisation
data = load_fasta_parallel("data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta", 200)


LmjF.01LmjF.02LmjF.03

LmjF.04

LmjF.05
LmjF.06
LmjF.07
LmjF.08
LmjF.09
LmjF.10
LmjF.11
LmjF.12
LmjF.13
LmjF.14
LmjF.15
LmjF.16
LmjF.17
LmjF.18
LmjF.19
LmjF.20
LmjF.21
LmjF.22
LmjF.23
LmjF.24
LmjF.25
LmjF.26
LmjF.27
LmjF.28
LmjF.29
LmjF.30
LmjF.31
LmjF.32
LmjF.33
LmjF.34
LmjF.35
LmjF.36


In [17]:
#Estiamtion + Détection

vecteur_debut = []
vecteur_fin = []
vecteur_longeur = []

df = pd.read_csv("data/Centromere_Positions/pos_libre.csv")
pos_libre = {
    int(row.Chromosome): (int(row.Start), int(row.End))
    for row in df.itertuples(index=False)
}

for i in range(0,36):
    temp_ =  data[i]['X3']*data[i]['X2']

    telo = 40000
    temp_ = temp_[telo:]

    mean = np.mean(temp_)

    peaks, properties = find_peaks(temp_,  prominence=mean*2, distance=20)

    window = 3000
    peak_medians = []

 



    #Moyenne
    peak_means = []
    for peak in peaks:
        start = max(0, peak - window//2)
        end = min(len(temp_), peak + window//2)
        mean_val = np.mean(temp_[start:end])
        peak_medians.append((peak, mean_val))
    

    # Trouver le pic avec la moyenne la plus élevée
    if peak_medians:
        peak_max, max_median = max(peak_medians, key=lambda x: x[1])
        max_index = peak_max




        # Appliquer un filtre pour lisser
        temp_X2_3 = uniform_filter1d(temp_, size=1500)
        #Zoom sur le pic
        temp_X2_3 = temp_X2_3[max_index-25000:max_index+25000]
        

        
        # Détection du nouveau pic
        peaks, properties = find_peaks(temp_X2_3, prominence=5, distance=200)

            
        # Trouver le pic avec la plus grande **prominence**
        if len(peaks) > 0:
            prominences = properties['prominences']
            peak_index = np.argmax(prominences)
            peak_max = peaks[peak_index]
        
            # Calcule la largeur du pic à mi-hauteur
            widths_result = peak_widths(temp_X2_3, peaks, rel_height=0.4)
        
            taille = widths_result[0][peak_index]
            seuil = widths_result[1][peak_index]
            debut = widths_result[2][peak_index]
            fin = widths_result[3][peak_index]


            debut = int(debut)
            fin = int(fin)

            #Additonner un pic qui est proche

            #Apres
            marge = 750
            finish = True
            distance = []
            temp_fin = []
            while finish:
                
                recherche = False
                limiteFin = min(len(temp_X2_3), fin + marge)
                pos = fin

                while pos < limiteFin and recherche == False:
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos += 1
                if recherche == True:
                     temp_fin.append(fin)
                     distance.append(0)
                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos += 1
                        distance[len(distance)-1] += 1

                     fin = pos
                else: 
                    finish = False

            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                fin = temp_fin[t]
                t = t -1 
            #Avant
            finish = True
            distance = 0
            distance = []
            temp_debut = []

            while finish:
                recherche = False
                limiteDebut = max(0, debut - marge)
                pos = debut
                while pos > limiteDebut and recherche == False:
 
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos -= 1
        
                if recherche == True:
                     temp_debut.append(debut)
                     distance.append(0)

                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos -= 1
                        distance[len(distance)-1] += 1
                     debut = pos
                else:
                    finish = False


            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                debut = temp_debut[t]
                t = t -1 
                

            taille = fin - debut


            


            #mise a niveau des position -> debut (position trouvé) + max_index-25000 (Zoom) +telo (postion coupé au debut) + 5000 (position coupé sur le calcule des metriques)
            debut = debut+max_index-25000+5000+telo
            fin = fin+max_index-25000+5000+telo
            debut = int(debut)
            fin = int(fin)
            max_index = max_index+5000+telo
            # Affichage


            #Regarder si on est loins des centromres trouvé chez L.major
            pos_start, pos_end = pos_libre[i+1]
            if abs(pos_start-debut) > 50000:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille} Attention !!! Diff position : {pos_start-debut}')
            else:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille}')




            vecteur_debut.append(debut)
            vecteur_fin.append(fin)
            vecteur_longeur.append(taille)


result = pd.DataFrame()
result['Chromosome'] = list(range(1, 37))
result['start'] = vecteur_debut 
result['end'] = vecteur_fin
result['length'] = vecteur_longeur

result.to_csv(f"major.csv", index=False)

1 : Commence 257127    Fini 258779  Taille : 1652
2 : Commence 265286    Fini 268347  Taille : 3061
3 : Commence 247037    Fini 253049  Taille : 6012
4 : Commence 125556    Fini 132214  Taille : 6658
5 : Commence 360714    Fini 367337  Taille : 6623
6 : Commence 122808    Fini 127353  Taille : 4545
7 : Commence 208630    Fini 214601  Taille : 5971
8 : Commence 487415    Fini 491520  Taille : 4105
9 : Commence 271664    Fini 277252  Taille : 5588
10 : Commence 296024    Fini 300542  Taille : 4518
11 : Commence 159688    Fini 161740  Taille : 2052
12 : Commence 287267    Fini 290888  Taille : 3621
13 : Commence 143539    Fini 145503  Taille : 1964
14 : Commence 158327    Fini 162610  Taille : 4283
15 : Commence 323047    Fini 329621  Taille : 6574
16 : Commence 340157    Fini 343824  Taille : 3667
17 : Commence 341042    Fini 344013  Taille : 2971
18 : Commence 443679    Fini 449382  Taille : 5703
19 : Commence 623659    Fini 630786  Taille : 7127
20 : Commence 523188    Fini 526438  Tai

# deep learning

In [ ]:
#Creation du Model
def get_true_positive(centromeres_,f_,size=3000,fail=[]):
    data = []
    for chrom in range(1,36+1):
        if chrom not in fail:
            start, stop = centromeres_[str(chrom)]
            if f_.names[0] == "1":
                seq = f_.sequences[f_.names.index(str(chrom))]
            else:
                seq = f_.sequences[chrom-1]
    
            if stop-start != 3000:
                stop = start + size
            # flip to get more positives
            data.append([one_hot_encoding(x) for x in seq[start:stop]])
            data.append([one_hot_encoding(x) for x in seq[start:stop][::-1]])
            

    return data

def get_true_negatives(centromeres_, lengths_, f_, N=1000, size=3000, seed=42):
    data = []
    positions = defaultdict(list)
    num_chromosomes = 36

    # Calcul taille totale
    total_length = 0
    chrom_lengths = {}
    oking = 0

    for chrom in range(1, num_chromosomes + 1):

        #Il y a 2 types de noms (1,2,3,4...) ou (LmjF.01,LmjF.02...)
        if f_.names[0] == "1":
            if str(chrom) in lengths_:
                chrom_len = lengths_[str(chrom)]
            else:
                chrom_len = -1
        else:
            if chrom != len(lengths_)+1:
                chrom_len = lengths_[chrom - 1]
            else:
                chrom_len = -1
        if chrom_len == -1:
            continue
        chrom_lengths[chrom] = chrom_len
        total_length += chrom_len

    # Calcul du nombre de séquences proportionnel à la taille
    for chrom, chrom_len in tqdm(chrom_lengths.items()):
        seq_count = int((chrom_len / total_length) * N)
        if seq_count == 0:
            seq_count = 1  # au moins une séquence

        start, stop = centromeres_[str(chrom)]
        tries = 0
        extracted = 0
        max_tries = seq_count * 10  # limite essais

        while extracted < seq_count and tries < max_tries:
            pos = random.randint(1, chrom_len - size)
            if oking == 0:
                print(pos)
                oking = 1 

            if pos > start and pos < stop:
                tries += 1
                continue  # exclure centromère
            positions[chrom].append(pos)
            extracted += 1
            tries += 1
    
    # Extraction des séquences
    testing = [0] * num_chromosomes
    for chrom in tqdm(positions.keys()):
        if f_.names[0] == "1":
            seq = f_.sequences[f_.names.index(str(chrom))]
        else:
            seq = f_.sequences[chrom - 1]

        for pos in positions[chrom]:
            testing[chrom - 1] += 1
            data.append([one_hot_encoding(x) for x in seq[pos:pos + size]])

    return data


In [ ]:


layers_values = [8,16,32,64]
layers_values = [32]

for layer in layers_values:
    for i in range(5,6):
        print("###############################################################################")
        print(f'{i}/10')
        print("###############################################################################")
     
        X_neg = []
        X_pos = []
        datas =     [["../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta","../../output/estimation/major/Major.csv",[]],
                    ["../../data/Fasta/genomes/Larabica/ragtag.scaffold.fasta","../../output/estimation/arabica.csv",[18,29]],
                    ["../../data/Fasta/genomes/Laethiopica/ragtag.scaffold.fasta","../../output/estimation/aethiopica.csv",[1,29]],
                    ["../../data/Fasta/genomes/Ltropica/LtropicaRef_genome.fasta","../../output/estimation/tropica.csv",[8,18]],
                    ["../../data/Fasta/genome.fasta","../../output/estimation/donovani/donovani_genome.csv",[]],
                    ["../../data/Fasta/genomes/Linfantum/TriTrypDB-68_LinfantumJPCM5_Genome.fasta","../../output/estimation/infantum.csv",[]],
                    ["../../data/Fasta/genomes/LamazonensisPH8/ragtag.scaffold.fasta","../../output/estimation/amazonensis.csv",[8,20,29,36]],
                    ["../../data/Fasta/genomes/Lmexicana/ragtag.scaffold.fasta","../../output/estimation/mexicana.csv",[8,20,29,36]],
                    ["../../data/Fasta/genomes/Ltarentolae/ragtag.scaffold.fasta","../../output/estimation/tarentolae.csv",[]],
                    ["../../data/Fasta/genomes/Lbraziliensis/ragtag.scaffold.fasta","../../output/estimation/braziliensis.csv",[20,34]],
                    ["../../data/Fasta/genomes/Lorientalis/ragtag.scaffold.fasta","../../output/estimation/orientalis.csv",[20,28]],
                    ["../../data/Fasta/genomes/Lmartiniquensis/ragtag.scaffold.fasta","../../output/estimation/martiniquensis.csv",[1]]]
        random.seed(i)
        for data in datas:
            print(f'Genome {data[1]}')
            f = FastA(data[0])
            centromeres = pd.read_csv(data[1])
            centromeres = {
                str(row['Chromosome']): (row['start'], row['end'])
                for _, row in centromeres.iterrows()
            }
    
            X_pos.extend(get_true_positive(centromeres,f,SIZE,data[2]))
            if f.names[0] == "1":
                lengths = f.get_lengths_as_dict()
            else:
                lengths = list(f.get_lengths_as_dict().values())
            X_neg.extend(get_true_negatives(centromeres,lengths,f,N=2500, size=SIZE,seed=i))
    
        
        print('##############################Data       ############################################')
        print(f'pos {len(X_pos)}')
        print(f'neg {len(X_neg)}')


        # Create label arrays
        y_pos = [1] * len(X_pos)
        y_neg = [0] * len(X_neg)
        
        # Combine and shuffle
        X = np.array(X_pos + X_neg)  # shape: (N, 3000, 4)
        y = np.array(y_pos + y_neg)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
    
    
        layer_size=layer
        print(layer_size)
        print(SIZE)

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        
        seed = 42
    
        tf.random.set_seed(seed)
        
        tf.config.experimental.enable_op_determinism()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=40,
            batch_size=32,
            class_weight={0: 1, 1: len(y_neg)/len(y_pos)},  # handle imbalance
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
        )
    
    
        y_pred = model.predict(X_test) > 0.2
        report = classification_report(y_test, y_pred, output_dict=True)
    
        metrics = {
            'f1_score': report['1']['f1-score'],
            'precision': report['1']['precision'],
            'recall': report['1']['recall']
        }





In [ ]:
#Utilisation du model
model = load_model('Scrypt/ia/model_32layer_seed5_all2.keras')

In [ ]:
import numpy as np
from tqdm import tqdm

def predict_sliding_windows(seq, model, window_size=3000, step=1500, batch_size=512):
    scores = []
    positions = []

    windows = []
    pos = []

    # Préparer toutes les fenêtres
    for i in range(0, len(seq) - window_size + 1, step):
        chunk = seq[i:i + window_size]
        chunk = [one_hot_encoding(x) for x in chunk]
        windows.append(chunk)
        pos.append((i, i + window_size))

        # Dès qu'on atteint un batch -> prédire
        if len(windows) == batch_size:
            X = np.array(windows).reshape(len(windows), window_size, 4)
            probs = model.predict(X, verbose=0).reshape(-1)
            scores.extend(probs)
            positions.extend(pos)
            windows, pos = [], []

    # Dernier batch
    if windows:
        X = np.array(windows).reshape(len(windows), window_size, 4)
        probs = model.predict(X, verbose=0).reshape(-1)
        scores.extend(probs)
        positions.extend(pos)

    return positions, scores


In [ ]:

datas =     [["../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta","../../output/estimation/major/Major.csv","major"],
            #["../../data/Fasta/genomes/Larabica/ragtag.scaffold.fasta","../../output/estimation/arabica.csv","arabica"],
            #["../../data/Fasta/genomes/Laethiopica/ragtag.scaffold.fasta","../../output/estimation/aethiopica.csv","aethiopica"],
            #["../../data/Fasta/genomes/Ltropica/LtropicaRef_genome.fasta","../../output/estimation/tropica.csv","tropica"],
            #["../../data/Fasta/genome.fasta","../../output/estimation/donovani/donovani_genome.csv",[]],
            #["../../data/Fasta/genomes/Linfantum/TriTrypDB-68_LinfantumJPCM5_Genome.fasta","../../output/estimation/infantum.csv","infantum"],
            #["../../data/Fasta/genomes/LamazonensisPH8/ragtag.scaffold.fasta","../../output/estimation/amazonensis.csv","amazonensis"],
            #["../../data/Fasta/genomes/Lmexicana/ragtag.scaffold.fasta","../../output/estimation/mexicana.csv","mexicana"],
            #["../../data/Fasta/genomes/Ltarentolae/ragtag.scaffold.fasta","../../output/estimation/tarentolae.csv","tarentolae"],
            #["../../data/Fasta/genomes/Lbraziliensis/ragtag.scaffold.fasta","../../output/estimation/braziliensis.csv","braziliensis"],
           # ["../../data/Fasta/genomes/Lpanamensis/TriTrypDB-68_LpanamensisMHOMPA94PSC1_Genome.fasta","../../output/estimation/panamensis.csv","panamensis"],]
            #["../../data/Fasta/genomes/Lorientalis/ragtag.scaffold.fasta","../../output/estimation/orientalis.csv","orientalis"],
            #["../../data/Fasta/genomes/Lmartiniquensis/ragtag.scaffold.fasta","../../output/estimation/martiniquensis.csv","martiniquensis"]]
            ]
for data in datas:
    vecteur_Debut = []
    vecteur_Fin = []
    vecteur_longeur = []

    f = FastA(data[0])
    centromeres = pd.read_csv(data[1])
    
    centromeres = {
        str(row['Chromosome']): (row['start'], row['end'])
        for _, row in centromeres.iterrows()
    }

    for chrom in range(1,37):
        print( chrom)           

        # chercher grossièrement sur tout le chromosome en avançant de 1500
        try:
            seq = f.sequences[f.names.index(str(chrom))]
        
        except:
            seq = f.sequences[chrom-1].upper().replace('N', 'T')
        clf()
        positions, scores = predict_sliding_windows(seq, model,3000,1500)
        _ = plot([x[0] for x in positions], scores)
        start, stop = centromeres[str(chrom)]
        axvline(start, color='r', alpha=0.5)
        axhline(0.2, color='g')

        dossier = Path(f"./test/{data[2]}/")
        dossier.mkdir(exist_ok=True)
        savefig(f"./test/{data[2]}/CNN1_{chrom}_L{data[2]}.png")
    
        positions_ini = positions
        scores_ini = scores


        #Zomm sur le plus grand pics
        seq = seq[positions[argmax(scores)][0]-10000:positions[argmax(scores)][1]+10000]
        #Utisation du model avec un pas de 50
        clf()
        positions, scores = predict_sliding_windows(seq, model,SIZE,50)
        _ = plot([x[0] for x in positions], scores)
        start, stop = centromeres[str(chrom)]
        
        #Dès que la moitié de la séquence (3000) est un centromère il le considère.
        # Il y a donc un décalage de 1500.
        
        start = start-1500
        stop = stop-1500
        
        axvline(start-positions_ini[argmax(scores_ini)][0]+10000, color='r', alpha=0.5)
        axhline(0.2, color='g')
        axvline(stop-positions_ini[argmax(scores_ini)][0]+10000, color='r', alpha=0.5)
        axhline(0.2, color='g')
        dossier = Path(f"./test/{data[2]}Estimation/")
        dossier.mkdir(exist_ok=True)
        savefig(f"./test/{data[2]}Estimation/CNN1_{chrom}_L{data[2]}.png")
    
    
        scores = np.array(scores)
        
        # Trouver le pic principal (avec un seuil si besoin)
        peaks, _ = find_peaks(scores, height=0.2)  # height = seuil minimal
        if len(peaks) != 0:
            main_peak = peaks[np.argmax(scores[peaks])]
            
            # Calcul de la largeur à mi-hauteur
            widths_result = peak_widths(scores, [main_peak], rel_height=0.5)

            # *50 car on a pris des pas de 50
            taille = widths_result[0][0]*50
            debut = widths_result[2][0]*50
            fin = widths_result[3][0]*50
        
            delta = positions_ini[argmax(scores_ini)][0]-10000 + 1500
            vecteur_Debut.append(debut+delta)
            vecteur_Fin.append(fin+delta)
            vecteur_longeur.append(taille)
        else:
            vecteur_Debut.append(0)
            vecteur_Fin.append(0)
            vecteur_longeur.append(0)
        
    result = pd.DataFrame()
    result['Chromosome'] = list(range(1, 37))
    result['start'] = vecteur_Debut 
    result['end'] = vecteur_Fin
    result['length'] = vecteur_longeur
    result.to_csv(f"./test/{data[2]}Estimation/{data[2]}.csv", index=False)
print("fin")